In [1]:
import numpy as np
import hls4ml
import os
from tensorflow import keras

DATA_DIR    = '/Users/giannalongo/Downloads/pioneer/476_Final'
OUTPUT_DIR  = '/Users/giannalongo/Downloads/pioneer/476_Final'
HLS_OUTPUT  = '/Users/giannalongo/Downloads/pioneer/476_Final/hls4ml_project_small'

model = keras.models.load_model(os.path.join(OUTPUT_DIR, 'cnn_model_small.h5'))
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv1d (Conv1D)             (None, 200, 4)            24        
                                                                 
 max_pooling1d (MaxPooling1D  (None, 100, 4)           0         
 )                                                               
                                                                 
 conv1d_1 (Conv1D)           (None, 100, 8)            168       
                                                                 
 max_pooling1d_1 (MaxPooling  (None, 50, 8)            0         
 1D)                                                             
                                                                 
 conv1d_2 (Conv1D)           (None, 50, 8)             200       
                                                                 
 global_average_pooling1d (G  (None, 8)                0

/Users/giannalongo/anaconda3/envs/hls4ml/lib/python3.10/site-packages/hls4ml/converters/__init__.py:27: UserWarning: WARNING: Pytorch converter is not enabled!
  warnings.warn("WARNING: Pytorch converter is not enabled!", stacklevel=1)


In [2]:
# Configure with aggressive quantization and high reuse factor
config = hls4ml.utils.config_from_keras_model(model, granularity='name')

# Use 12-bit fixed point instead of 16-bit
for layer in config['LayerName']:
    for key in config['LayerName'][layer]['Precision']:
        config['LayerName'][layer]['Precision'][key] = 'fixed<24,8>'

config['Model']['Precision']    = 'fixed<24,8>'
config['Model']['ReuseFactor']  = 4

import pprint
pprint.pprint(config)

Interpreting Sequential
Topology:
Layer name: conv1d_input, layer type: InputLayer, input shapes: [[None, 200, 1]], output shape: [None, 200, 1]
Layer name: conv1d, layer type: Conv1D, input shapes: [[None, 200, 1]], output shape: [None, 200, 4]
Layer name: max_pooling1d, layer type: MaxPooling1D, input shapes: [[None, 200, 4]], output shape: [None, 100, 4]
Layer name: conv1d_1, layer type: Conv1D, input shapes: [[None, 100, 4]], output shape: [None, 100, 8]
Layer name: max_pooling1d_1, layer type: MaxPooling1D, input shapes: [[None, 100, 8]], output shape: [None, 50, 8]
Layer name: conv1d_2, layer type: Conv1D, input shapes: [[None, 50, 8]], output shape: [None, 50, 8]
Layer name: global_average_pooling1d, layer type: GlobalAveragePooling1D, input shapes: [[None, 50, 8]], output shape: [None, 8]
Layer name: dense, layer type: Dense, input shapes: [[None, 8]], output shape: [None, 16]
Layer name: dense_1, layer type: Dense, input shapes: [[None, 16]], output shape: [None, 3]
{'LayerNam

In [3]:
import shutil

if os.path.exists(HLS_OUTPUT):
    shutil.rmtree(HLS_OUTPUT)

hls_model = hls4ml.converters.convert_from_keras_model(
    model,
    hls_config=config,
    output_dir=HLS_OUTPUT,
    backend='Vitis',
    part='xc7a200tsbg484-1',
    clock_period=10
)

hls_model.compile()

X_test = np.load(os.path.join(OUTPUT_DIR, 'X_test_small.npy'))
Y_test = np.load(os.path.join(OUTPUT_DIR, 'Y_test_small.npy'))

y_keras = np.argmax(model.predict(X_test), axis=1)
y_hls   = np.argmax(hls_model.predict(X_test), axis=1)

match = np.mean(y_keras == y_hls)
print(f'Keras vs HLS agreement: {match*100:.2f}%')

Interpreting Sequential
Topology:
Layer name: conv1d_input, layer type: InputLayer, input shapes: [[None, 200, 1]], output shape: [None, 200, 1]
Layer name: conv1d, layer type: Conv1D, input shapes: [[None, 200, 1]], output shape: [None, 200, 4]
Layer name: max_pooling1d, layer type: MaxPooling1D, input shapes: [[None, 200, 4]], output shape: [None, 100, 4]
Layer name: conv1d_1, layer type: Conv1D, input shapes: [[None, 100, 4]], output shape: [None, 100, 8]
Layer name: max_pooling1d_1, layer type: MaxPooling1D, input shapes: [[None, 100, 8]], output shape: [None, 50, 8]
Layer name: conv1d_2, layer type: Conv1D, input shapes: [[None, 50, 8]], output shape: [None, 50, 8]
Layer name: global_average_pooling1d, layer type: GlobalAveragePooling1D, input shapes: [[None, 50, 8]], output shape: [None, 8]
Layer name: dense, layer type: Dense, input shapes: [[None, 8]], output shape: [None, 16]
Layer name: dense_1, layer type: Dense, input shapes: [[None, 16]], output shape: [None, 3]
Creating H

2026-05-07 14:13:44.318614: W tensorflow/tsl/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz


Keras vs HLS agreement: 99.87%


In [4]:
# Save test data and generate tb_data files
TB_DIR = os.path.join(HLS_OUTPUT, 'tb_data')
os.makedirs(TB_DIR, exist_ok=True)

X_flat = X_test.reshape(X_test.shape[0], -1)
Y_pred = model.predict(X_test)

with open(os.path.join(TB_DIR, 'tb_input_features.dat'), 'w') as f:
    for row in X_flat:
        f.write(' '.join([str(x) for x in row]) + '\n')

with open(os.path.join(TB_DIR, 'tb_output_predictions.dat'), 'w') as f:
    for row in Y_pred:
        f.write(' '.join([str(x) for x in row]) + '\n')

np.save(os.path.join(OUTPUT_DIR, 'X_test_small.npy'), X_test)
np.save(os.path.join(OUTPUT_DIR, 'Y_test_small.npy'), Y_test)

print('Done. tb_data files saved.')

24/24 [==============================] - 0s 599us/step
Done. tb_data files saved.


In [5]:
import pandas as pd

rows = []
for layer in model.layers:
    config_layer = layer.get_config()
    params = int(layer.count_params())
    layer_type = layer.__class__.__name__
    in_shape = layer.input_shape
    in_channels = int(in_shape[-1]) if not isinstance(in_shape[-1], tuple) else 0

    dsps = 0
    if layer_type == 'Conv1D':
        kernel_size = config_layer['kernel_size'][0]
        dsps = int(kernel_size) * int(config_layer['filters']) * in_channels
    elif layer_type == 'Dense':
        dsps = in_channels * int(config_layer['units'])

    # Apply ReuseFactor
    dsps = dsps // 4

    rows.append({
        'Layer': layer.name,
        'Type': layer_type,
        'Output Shape': str(layer.output_shape),
        'Parameters': params,
        'Est. DSPs (RF=4)': dsps
    })

df = pd.DataFrame(rows)
total_dsps = sum(r['Est. DSPs (RF=4)'] for r in rows)

print(df.to_string())
print(f'\nTotal parameters: {model.count_params()}')
print(f'Total estimated DSPs (ReuseFactor=4): {total_dsps}')
print(f'Artix-7 XC7A200T DSP limit: 740')
print(f'Usage: {total_dsps/740*100:.1f}%')

                      Layer                    Type    Output Shape  Parameters  Est. DSPs (RF=4)
0                    conv1d                  Conv1D  (None, 200, 4)          24                 5
1             max_pooling1d            MaxPooling1D  (None, 100, 4)           0                 0
2                  conv1d_1                  Conv1D  (None, 100, 8)         168                40
3           max_pooling1d_1            MaxPooling1D   (None, 50, 8)           0                 0
4                  conv1d_2                  Conv1D   (None, 50, 8)         200                48
5  global_average_pooling1d  GlobalAveragePooling1D       (None, 8)           0                 0
6                     dense                   Dense      (None, 16)         144                32
7                   dense_1                   Dense       (None, 3)          51                12

Total parameters: 587
Total estimated DSPs (ReuseFactor=4): 137
Artix-7 XC7A200T DSP limit: 740
Usage: 18.5%
